# Basic data cleaning and tokenization

## Cleaning

Some simple, regex-based cleaning is performed on train and dev datasets, e.g. to remove HTML tags from Wikipedia articles, non-verbal cues from subtitles, or even to correct I’s that were incorrectly recognized as l’s in OCR’ed uppercase text.

In [3]:
from pathlib import Path
from mrclean import *
DATA_ROOT = Path("./")
from pathlib import Path
from tokenizers import (Tokenizer, decoders, models, pre_tokenizers,
                        processors, trainers)
from tokenizers.normalizers import NFKC

In [ ]:
SEQ_LENGTH = 128 # this is a legacy parameter, it does not affect cleaning
DATA_SPLITS = ['babylm_10M', 'babylm_dev']

CLEANUP_FUNCTIONS = {
    'childes': cleanup_aochildes,
    'bnc_spoken': cleanup_bnc_spoken,
    'cbt': cleanup_cbt,
    'children_stories': cleanup_children_stories,
    'gutenberg': cleanup_gutenberg,
    'open_subtitles': cleanup_open_subtitles,
    'qed': cleanup_qed,
    'simple_wiki': cleanup_simple_wikipedia,
    'switchboard': cleanup_switchboard,
    'wikipedia': cleanup_wikipedia,
}


In [10]:
for split in DATA_SPLITS:
    INPUT_DIR = DATA_ROOT / 'data' / split
    OUTPUT_DIR = DATA_ROOT / 'data' / f'{split}_clean'
    
    OUTPUT_DIR.mkdir(exist_ok=True)

    train_files = [f for f in INPUT_DIR.iterdir() if f.is_file() and f.suffix in ['.train', '.dev']]
    
    for file in train_files:
        text = file.read_text()
        print(file.stem)
        cleaned_text = CLEANUP_FUNCTIONS[file.stem](text, SEQ_LENGTH)
        (OUTPUT_DIR / file.name).write_text(cleaned_text)
        print(f"🧹 Cleaned '{file.name}' (size {len(text)} -> {len(cleaned_text)}) in {split}")


childes
🧹 Cleaned 'childes.train' (size 15482927 -> 15482733) in babylm_10M
switchboard
🧹 Cleaned 'switchboard.train' (size 719322 -> 719322) in babylm_10M
bnc_spoken
🧹 Cleaned 'bnc_spoken.train' (size 4883879 -> 4851676) in babylm_10M
gutenberg
🧹 Cleaned 'gutenberg.train' (size 13910986 -> 13910986) in babylm_10M
open_subtitles
🧹 Cleaned 'open_subtitles.train' (size 10806305 -> 10804026) in babylm_10M
simple_wiki
🧹 Cleaned 'simple_wiki.train' (size 8411630 -> 8387062) in babylm_10M


## Training a tokenizer

In [16]:
# We train the tokenizer on the train data only
data_dir = Path("./data/babylm_10M_clean/")

paths = [str(f) for f in data_dir.glob("*") if f.is_file() and not f.name.endswith(".DS_Store") and f.suffix in [".train"]]

# paths
print(len(paths))
assert len(paths) > 0, 'No data files found'

6


In [17]:
tokenizer = Tokenizer(models.BPE())

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
tokenizer.decoder = decoders.ByteLevel()
tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
tokenizer.normalizer = NFKC()

In [19]:
# trainer = trainers.BpeTrainer(vocab_size=16000, min_frequency=2, special_tokens=["<pad>", "<s>", "</s>"])
# tokenizer.train(paths, trainer)
from tokenizers import trainers

# (2) Reserve exactly 50 257 tokens total: 256 bytes + 50 000 merges + 1 special token
trainer = trainers.BpeTrainer(
    vocab_size=50257,                   # total token count
    min_frequency=4,                    # ignore subwords occurring fewer than twice
    special_tokens=["<|endoftext|>"]    # reserve 1 slot (ID 50256) for end-of-text
)
tokenizer.train(paths, trainer)


In [20]:
print(DATA_ROOT)

.


In [21]:
tokenizer_path =  DATA_ROOT / "models/tokenizer16000.json"
tokenizer_path =  DATA_ROOT / "models/tokenizer50257.json"


In [5]:
tokenizer_path

PosixPath('models/tokenizer50257.json')

In [ ]:
# tokenizer.save(str(tokenizer_path), pretty=True)

## Testing the tokenizer

In [24]:

tokenizer = Tokenizer.from_file(str(tokenizer_path))


# text = 'Shiro Okada (岡田志郎, "Okada Shirō", June 9, 1949; Hirakata, Osaka {age 71} - ) is a Japanese guitarist who participate in the Group Sound band, the Ox. His nickname was Shiro (シロー) and his real name is Shiro Okamoto (岡田史郎).'
text = "The quick brown fox jumps over the lazy dog. "

encoded = tokenizer.encode(text)
print(f"Encoded String: {encoded.tokens}")

print(f"Encoded IDs: {encoded.ids}")

decoded = tokenizer.decode(encoded.ids)
print(f"Decoded String: {decoded}")


Encoded String: ['ĠThe', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.', 'Ġ']
Encoded IDs: [300, 1671, 3080, 5571, 15480, 536, 186, 11331, 1408, 14, 144]
Decoded String:  The quick brown fox jumps over the lazy dog. 


In [2]:
#!/usr/bin/env python3 
import pandas as pd
import os
import torch
os.getcwd()


training_location = '/home/jorge/tokenPred/babylm_10m/train_files/CustomLlama/data/babylm_10M_clean'
training_files = os.listdir(training_location)
# print(training_files)
# we write a dictionary mapping the file name to the file location
file_dict = {training_files[i][:-6]: training_location + '/' + training_files[i] 
             for i in range(len(training_files)) if training_files[i].endswith('.train')}
file_dict

file_text = {}
#we read the files and store them in a dictionary
for key in file_dict.keys():
    with open(file_dict[key],'r') as file:
        # we skip the files that are pickle files
            try:
                # we read the file. If it's not a pickle file, we skip it
                file_text[key] = file.read()   
                print(len(file_text[key]), key) 
            except:
                pass

print(file_text.keys())

corpus = ''
corpus = corpus.join(file_text.values())

15482733 childes
719322 switchboard
4851676 bnc_spoken
13910986 gutenberg
10804026 open_subtitles
8387062 simple_wiki
dict_keys(['childes', 'switchboard', 'bnc_spoken', 'gutenberg', 'open_subtitles', 'simple_wiki'])


In [34]:
tokenized_text = tokenizer.encode(corpus)

In [35]:
len(tokenized_text.ids)

15513496

In [43]:
import pickle
# Save the corpus to a pickle file
with open('/home/jorge/tokenPred/babylm_10m/train_files/CustomLlama/data/babylm_10M_clean/gptencoded.pkl', 'wb') as f:
    pickle.dump(tokenized_text, f)

In [ ]:
pickled_files = '/home/jorge/tokenPred/babylm_10m/train_files/CustomLlama/data/babylm_10M_clean/gptencoded.pkl'
with open(
    pickled_files, "rb"
) as f:
    tokenized_files = pickle.load(f)
print("Loaded tokenized files.")

Loaded tokenized files.


In [1]:
#!/usr/bin/env python3
import pickle
import torch
from transformers import AutoTokenizer

# Use GPT-2 tokenizer to match the repository
tokenizer = AutoTokenizer.from_pretrained("gpt2")
MODEL_EOS = 50256  # GPT-2's EOS token ID

# Your existing file reading code
training_location = '/home/jorge/tokenPred/babylm_10m/train_files/CustomLlama/data/babylm_10M_clean'
# ... [keep your file reading code as is] ...

# NEW: Chunk the data like the repository does
CHUNK_SIZE = 512  # Match your BLOCK_SIZE
chunked_data = []

for key in file_text.keys():
    # Tokenize each file separately
    tokens = tokenizer(text=file_text[key])['input_ids']
    
    # Chunk into fixed sizes
    num_chunks = len(tokens) // CHUNK_SIZE
    for i in range(num_chunks):
        start = i * CHUNK_SIZE
        end = (i + 1) * CHUNK_SIZE
        chunk = tokens[start:end]
        # Add EOS tokens around each chunk
        chunked_data.append([MODEL_EOS] + chunk + [MODEL_EOS])



NameError: name 'file_text' is not defined

In [ ]:
# Save the chunked data
with open('/home/jorge/tokenPred/babylm_10m/train_files/CustomLlama/data/babylm_10M_clean/gpt_chunked.pkl', 'wb') as f:
    pickle.dump(chunked_data, f)

In [ ]:
tokenizer.encode('<|endoftext|>').ids

[0]

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class ChunkedDataset(torch.utils.data.Dataset):
    def __init__(self, chunked_data):
        self.data = chunked_data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return torch.LongTensor(self.data[idx])

def collate_fn(batch):
    MODEL_EOS = 50256
    # Pad sequences with EOS token
    tokens = pad_sequence(batch, padding_value=MODEL_EOS, batch_first=True)
    # Split into input and target
    input_tokens = tokens[:, :-1]
    target_tokens = tokens[:, 1:]
    # Create mask (False for EOS padding, True for real tokens)
    target_mask = input_tokens != MODEL_EOS
    target_mask[:, 0] = 1  # Always include first token
    
    return input_tokens, target_tokens, target_mask

In [5]:
#we load up gpt-berts text file called baby cosmo fine 10m. We will see how the BabyLM 10m GPT-2 model performs
#on a dataset (FineWeb) consisting of fine quality website as well as synthetically generated text from mixtral 8x7b.
import pandas as pd

df = pd.read_json("hf://datasets/ltg/babylm-2024-baby-cosmo-fine-10m/train.jsonl", lines=True)


In [6]:
import regex as re
import html, unicodedata

def clean_text(text: str) -> str:
    if not text:
        return ""
    # normalize line endings early
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # fix common encoding junk
    text = text.replace("\u00A0", " ")        # NBSP → space
    # unescape HTML entities
    text = html.unescape(text)
    # strip very likely HTML tags (conservative)
    text = re.sub(r"<[^>\n]{1,200}>", "", text)
    # drop bare URLs (optional)
    text = re.sub(r"(https?://\S+|www\.\S+)", "", text)
    # NFKC unicode normalization (keeps accents, smart quotes, emojis, etc.)
    text = unicodedata.normalize("NFKC", text)
    # remove control chars except tab/newline
    text = re.sub(r"[\p{Cc}\p{Cf}&&[^\n\t]]+", "", text)
    # collapse horizontal whitespace; keep paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()
    # ensure a trailing newline (nice for concatenation/packing)
    if text and not text.endswith("\n"):
        text += "\n"
    return text

trial_text = 'venues are very cool\n.'
df['cleaned_text'] = df['text'].apply(clean_text)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
MODEL_EOS = tokenizer.eos_token_id
CHUNK_SIZE = 512  # Match your BLOCK_SIZE
chunked_data = []
# we tokenize and chunk the cleaned text
for text in df['cleaned_text']:
    # Tokenize each file separately
    tokens = tokenizer(text=text)['input_ids']
    
    # Chunk into fixed sizes
    num_chunks = len(tokens) // CHUNK_SIZE
    for i in range(num_chunks):
        start = i * CHUNK_SIZE
        end = (i + 1) * CHUNK_SIZE
        chunk = tokens[start:end]
        # Add EOS tokens around each chunk
        chunked_data.append([MODEL_EOS] + chunk + [MODEL_EOS])
print(f"Created {len(chunked_data)} chunks from cleaned babyCosmosFineWeb dataset.")

Token indices sequence length is longer than the specified maximum sequence length for this model (8910 > 1024). Running this sequence through the model will result in indexing errors


In [15]:
import pickle
#we save the chunked data as a pickle file, naming it babycosmofineweb_chunked.pkl
with open('/home/jorge/tokenPred/babylm_10m/train_files/CustomLlama/data/babylm_10M_clean/babycosmofineweb_chunked.pkl', 'wb') as f:
    pickle.dump(chunked_data, f)

In [17]:
# we load the chunked data back to verify
with open('/home/jorge/tokenPred/babylm_10m/train_files/CustomLlama/data/babylm_10M_clean/babycosmofineweb_chunked.pkl', 'rb') as f:
    loaded_chunked_data = pickle.load(f)
tokenizer.decode(loaded_chunked_data[0])

'<|endoftext|>In the bustling coastal town of Blackpool, nestled between the Irish Sea and the picturesque hills of Lancashire, two pillars of English football stood tall - A.F.C. Blackpool and Blackpool F.C. And at the heart of these institutions was a man who wore many hats; Stuart Parker, a seasoned footballer, found himself juggling dual responsibilities as a manager for A.FC. Blackpool and an influential figure at Blackpool F.C.\n\nOne sunny afternoon, while engrossed in his tactical board markings, a knock echoed through the modest A.F.C. Blackpool headquarters. Stuart looked up from his scribbled formations to find the eager face of Jamie, a young striker with dreams bigger than his boots.\n\n"Coach Parker," panted Jamie, breathless from excitement rather than exertion, "I\'ve heard about your work here and at Blackpool FC. Could you teach me how to become a better player?"\n\nAh, thought Stuart, recognizing an opportunity to impart wisdom earned over years spent on pitches just

In [23]:
sub_df = df.sample(n=10)
#from printing the text, we can see the text lengths vary significantly.

for i, row in sub_df.iterrows():
    print(f"--- Sample {i} ---")
    print(row['text'])
    print(len(row['text']))


--- Sample 1914 ---
Creating Modern Probability
Its Mathematics, Physics and Philosophy in Historical Perspective
Normal Price: $185.00
Your Price: $166.50 AUD, inc. GST
Shipping: $7.95 per order
You Save: $18.50! (10% off normal price)
Plus...earn $8.33 in Boomerang Bucks
Availability: Available to Backorder, No Due Date for Supply, Not for Xmas
Creating Modern Probability by Jan von Plato
Book DescriptionThis is the only book to chart the history and development of modern probability theory. It shows how in the first thirty years of this century probability theory became a mathematical science. The author also traces the development of probabilistic concepts and theories in statistical and quantum physics. There are chapters dealing with chance phenomena, as well as the main mathematical theories of today, together with their foundational and philosophical problems. Among the theorists whose work is treated at some length are Kolmogorov, von Mises and de Finetti. The principal audien